In [8]:
# Install dependencies
!pip install gymnasium[atari] stable-baselines3[extra] ale-py opencv-python

In [9]:
# Environment setup
import gymnasium as gym
import ale_py
import numpy as np

from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

gym.register_envs(ale_py)

def make_env():
    env = gym.make("ALE/Skiing-v5")
    env = AtariWrapper(env)
    return env

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)

In [10]:

import gymnasium as gym
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from stable_baselines3.common.atari_wrappers import AtariWrapper


In [11]:

def make_env():
    env = gym.make("ALE/Skiing-v5")
    env = AtariWrapper(env)
    return env


In [12]:

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)
env = VecTransposeImage(env)


In [13]:

try:
    model = DQN.load("skiing_cnn_dqn", env=env)
    print("Loaded existing model")

    try:
        model.load_replay_buffer("replay_buffer.pkl")
        print("Loaded replay buffer")
    except:
        print("No replay buffer found")

except:
    print("Creating new model")
    model = DQN(
        "CnnPolicy",
        env,
        verbose=1,
        device="cuda"  # use GPU if available
    )


Creating new model
Using cpu device


C:\Users\gabea\anaconda3\Lib\site-packages\stable_baselines3\common\buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 56.46GB > 47.37GB
  warnings.warn(


In [ ]:

for i in range(5):  # train in chunks
    model.learn(
        total_timesteps=100_000,
        reset_num_timesteps=False
    )

    model.save("skiing_cnn_dqn")
    model.save_replay_buffer("replay_buffer.pkl")

    print(f"Checkpoint {i} saved")


In [ ]:

def make_env_render():
    env = gym.make("ALE/Skiing-v5", render_mode="human")
    env = AtariWrapper(env)
    return env

env = DummyVecEnv([make_env_render])
env = VecFrameStack(env, n_stack=4)
env = VecTransposeImage(env)

model = DQN.load("skiing_cnn_dqn", env=env)


In [ ]:

obs = env.reset()

while True:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = env.step(action)

    if done:
        obs = env.reset()
